# 1) Imports básicos


In [55]:
import io
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.linear_model import LinearRegression

# 2) Upload do CSV de sales_items


In [56]:
sales_items = pd.read_csv(io.BytesIO(uploaded['dataset_fashion_store_salesitems.csv']))
sales_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2253 entries, 0 to 2252
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   item_id            2253 non-null   int64  
 1   sale_id            2253 non-null   int64  
 2   product_id         2253 non-null   int64  
 3   quantity           2253 non-null   int64  
 4   original_price     2253 non-null   float64
 5   unit_price         2253 non-null   float64
 6   discount_applied   2253 non-null   float64
 7   discount_percent   2253 non-null   object 
 8   discounted         2253 non-null   int64  
 9   item_total         2253 non-null   float64
 10  sale_date          2253 non-null   object 
 11  channel            2253 non-null   object 
 12  channel_campaigns  2253 non-null   object 
dtypes: float64(4), int64(5), object(4)
memory usage: 228.9+ KB


# 3) Garantir que a coluna de data está como datetime


In [57]:
sales_items['sale_date'] = pd.to_datetime(sales_items['sale_date'])

# 4) Definir a coluna de valor da linha


In [58]:
sales_items['line_value'] = sales_items['item_total']

sales_items[['sale_date', 'line_value']].head()

,sale_date,line_value
0,2025-06-16,81.80
1,2025-06-17,81.79
2,2025-04-16,80.76
3,2025-05-06,78.52
4,2025-06-15,78.52


# 5) Agregar faturamento diário


In [59]:
daily_raw = (
    sales_items
    .groupby('sale_date', as_index=False)['line_value']
    .sum()
    .rename(columns={'line_value': 'total_sales'})
)

daily_raw = daily.sort_values('sale_date')
daily_raw.head()

,sale_date,total_sales,t
0,2025-04-04,5464.76,1
1,2025-04-05,6370.74,2
2,2025-04-06,4069.68,3
3,2025-04-07,0.00,4
4,2025-04-08,0.00,5


## 5.1) Criar um calendário contínuo de datas


In [60]:
full_dates = pd.DataFrame({
    'sale_date': pd.date_range(
        start=daily_raw['sale_date'].min(),
        end=daily_raw['sale_date'].max(),
        freq='D'
    )
})

## 5.2) Fazer merge para incluir dias sem venda (como 0)


In [61]:
daily = full_dates.merge(daily_raw, on='sale_date', how='left')
daily['total_sales'] = daily['total_sales'].fillna(0)

daily.head()

,sale_date,total_sales,t
0,2025-04-04,5464.76,1
1,2025-04-05,6370.74,2
2,2025-04-06,4069.68,3
3,2025-04-07,0.00,4
4,2025-04-08,0.00,5


# 6) Criar índice temporal e treinar regressão linear simples


In [62]:
daily['t'] = np.arange(1, len(daily) + 1)

X = daily[['t']].values
y = daily['total_sales'].values

model = LinearRegression()
model.fit(X, y)

LinearRegression()

# 7) Gerar previsão para os próximos N dias


In [63]:
n_forecast_days = 7

last_t = daily['t'].iloc[-1]
future_t = np.arange(last_t + 1, last_t + 1 + n_forecast_days)

last_date = daily['sale_date'].iloc[-1]
future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=n_forecast_days,
    freq='D'
)

future_sales = model.predict(future_t.reshape(-1, 1))

# 8) Montar tabela final histórico + previsão


In [64]:
hist = daily[['sale_date', 'total_sales']].copy()
hist['forecast_sales'] = np.nan

future = pd.DataFrame({
    'sale_date': future_dates,
    'total_sales': np.nan,
    'forecast_sales': future_sales
})

forecast_df = pd.concat([hist, future], ignore_index=True)
forecast_df.tail()

,sale_date,total_sales,forecast_sales
77,2025-06-20,NaN,4283.365982
78,2025-06-21,NaN,4282.371244
79,2025-06-22,NaN,4281.376507
80,2025-06-23,NaN,4280.381770
81,2025-06-24,NaN,4279.387033


# 9) Exportar CSV para usar no Power BI


In [65]:
forecast_df.to_csv('forecast_sales.csv', index=False)

files.download('forecast_sales.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>